# A 3D view of the docking

Plots describe a simulation one quantity at a time: a trajectory in a plane, an error against time, a speed against distance. None of them shows the approach as a whole, with the deputy turning to keep the chief in view, firing its thrusters, and sliding along the corridor toward the port while the Earth passes below and the Sun sets behind it. This notebook exports the result of a closed-loop simulation to a 3D viewer that plays it back.

The viewer is a single HTML file. It contains the data of the run and the code that draws it, and it opens in any recent browser with no server and nothing to install. It displays the simulation and does not compute anything: whatever it shows was decided by the simulator, so the pictures and the numbers of the other notebooks are two views of the same run.

In [1]:
from pathlib import Path

import numpy as np
from IPython.display import IFrame

from ios_mission_planner.sim.config import SimulationConfig
from ios_mission_planner.sim.simulator import simulate
from ios_mission_planner.viz.viewer import LVLH_TO_SCENE, build_viewer_data, export_viewer

result = simulate(SimulationConfig(seed=1))
print(f"{result.outcome} after {result.duration / 60:.1f} min, {len(result.t)} steps")

docked after 96.0 min, 5760 steps


## Exporting the viewer

`export_viewer` takes a `SimulationResult` and writes the file. Before writing it reduces the run to what the playback needs, with `build_viewer_data`: the true relative position and attitude of the deputy, the thrust on each body axis, the navigation estimate, the direction of the Sun and the eclipse flag, and the numbers shown on screen. Far from the chief one sample every five seconds is enough; inside the last 100 m every step of the simulation is kept, so the final approach is played back at full resolution.

In [2]:
path = export_viewer(result, "docking_viewer.html", title="Docking of the reference mission")
data = build_viewer_data(result)

print(f"{path.name}: {path.stat().st_size / 1e3:.0f} kB, {len(data['t'])} samples of {len(result.t)} steps")
IFrame(path.name, width="100%", height=640)

docking_viewer.html: 374 kB, 2246 samples of 5760 steps


## What the viewer shows

The scene is drawn in the LVLH frame of the chief, the same frame as every relative-motion plot of this book. The chief is fixed at the center, with its docking port looking backward along the V-bar, and the direction of flight is to the right. The Earth is below and turns underneath, once per orbit, because in this frame it is the rest of the universe that rotates. The light comes from the direction in which the Sun was at each instant of the simulation, computed by the illumination module, and it goes out when the chief enters the shadow of the Earth.

The yellow line is the trajectory already flown, on top of the fainter line of the whole trajectory. The purple wireframe is the approach corridor that the predictive controller respects. The green cone is the field of view of the relative sensor, and the small green wire sphere is the position that the navigation believes the deputy has: far from the chief it is visibly separate from the deputy, and it converges onto it as the range decreases. A plume appears on a face of the deputy whenever the thrusters on that axis fire.

The panel on the left gives the phase of the mission, the distance and the closing speed between the two ports, the lateral offset, the attitude error, the velocity change spent so far, the fraction of the wheel capacity in use and whether the chief is in sunlight. The bar at the bottom is the timeline, colored by phase, and can be dragged to any instant. When the run ends, a panel on the right lists the conditions at contact against their limits.

Four cameras are available. The overview frames both spacecraft and zooms in as they get closer; dragging rotates it and the wheel changes the distance. The side view looks along the orbit normal and reproduces the plane of the trajectory plots. The chase view follows the deputy from behind. The port camera looks from the front of the deputy along the axis of its relative sensor, which is the picture that an optical sensor would see: the target next to the port of the chief stays under the crosshair for the whole approach, because the attitude control keeps it there.

The state of the viewer can be set in the address of the page, which makes it possible to link to a specific moment: `docking_viewer.html?t=5600&cam=port&play=0` opens the port camera at 5600 s with the playback paused. The parameters are `t` (time in seconds), `cam` (`overview`, `side`, `chase` or `port`), `play` (0 or 1), `speed`, and `az`, `el`, `zoom` for the overview camera.

## The frame of the scene

A 3D library has its own axes, with $Y$ up. The scene uses $(X, Y, Z) = (y, x, -z)$ of the LVLH frame: the along-track direction to the right, the radial direction up, and the opposite of the orbit normal toward the viewer. This mapping is a rotation, with determinant $+1$, so it preserves the handedness of everything it transforms. The attitude of the deputy is exported as the quaternion of the rotation from its body axes to the scene axes, in the active convention of graphics libraries, which is the transpose of the inertial-to-body convention of the attitude chapters; `rotation_to_scene_quaternion` exists to keep that conversion in one tested place.

The cell below checks the exported data against the simulation at the instant of contact: the body $x$ axis of the deputy, which carries its port, must point along the direction of flight, toward the port of the chief, and its center of mass must be about 1.6 m behind the origin, the sum of the two port offsets, less the few centimetres travelled during the last step of the simulation.

In [3]:
def rotate(q, v):
    """Rotation of a vector by a quaternion (x, y, z, w), as the viewer applies it."""
    axis, w = np.array(q[:3]), q[3]
    return v + 2.0 * np.cross(axis, np.cross(axis, v) + w * v)

last_q = data["q"][-4:]
last_p = np.array(data["p"][-3:])

print("determinant of the LVLH-to-scene mapping:", np.linalg.det(LVLH_TO_SCENE))
print("body x axis in the scene at contact:    ", np.round(rotate(last_q, np.array([1.0, 0.0, 0.0])), 4))
print("body z axis in the scene at contact:    ", np.round(rotate(last_q, np.array([0.0, 0.0, 1.0])), 4))
print("position of the deputy at contact [m]:  ", np.round(last_p, 3), "(radial, along-track, cross-track)")

determinant of the LVLH-to-scene mapping: 1.0
body x axis in the scene at contact:     [ 1. -0.  0.]
body z axis in the scene at contact:     [-0. -0. -1.]
position of the deputy at contact [m]:   [ 0.000e+00 -1.551e+00  1.000e-03] (radial, along-track, cross-track)


The body $x$ axis is along the scene $X$, the direction of flight, and the body $z$ axis along $-Z$, which is the orbit normal: the deputy is docked with the attitude that the guidance commanded, and the viewer draws it there.

## A second case: the avoidance maneuver

The same export works for any run. The one below repeats the failure of the closed-loop notebook, the loss of the relative sensor during the final approach, and shows the collision avoidance maneuver in 3D: the burn away from the chief, visible as a plume on the front of the deputy, and the departure below and then ahead of the chief.

In [4]:
failed = SimulationConfig(seed=1)
start_final = next(time for time, label, _ in result.events if label == "final approach")
failed.failures.sensor_outage_start = start_final + 160.0
failed.failures.sensor_outage_duration = 1e5
failed.guidance.abort_coast_duration = 1500.0
aborted = simulate(failed)

abort_path = export_viewer(aborted, "abort_viewer.html", title="Loss of the relative sensor and avoidance maneuver")
print(f"{aborted.outcome} ({aborted.abort_reason}); {abort_path.name}: {abort_path.stat().st_size / 1e3:.0f} kB")
IFrame(abort_path.name + f"?t={start_final + 150:.0f}&speed=10", width="100%", height=640)

aborted (relative measurements lost); abort_viewer.html: 501 kB


## Summary

`export_viewer` turns a `SimulationResult` into a self-contained page that replays the approach in three dimensions, in the frame of the chief, with the lighting of the simulated epoch, the trajectory, the corridor, the field of view of the sensor, the navigation estimate, the thruster firings and the main numbers of the approach. It adds no result of its own. Its purpose is to make the behaviour of the whole system visible at once, and to show a mission to someone who will not read forty plots.

Three limits should be stated. The two spacecraft are schematic shapes with the right dimensions of the ports and of the sensor geometry, not models of real vehicles, and the Earth is a decorative sphere without real geography. The 3D library is loaded from a public content delivery network, so the page needs a connection the first time it is opened. And the viewer plays back the truth of the simulation: what it shows is exactly as right as the simulation behind it.